In [ ]:
# 我觉得搜索模块不太对劲，在这里打印所有的搜索结果，不要删除后的版本，不要走完所有node

from lib.agentic.graph import AgenticGraph
from lib.agentic.config import get_agent_state_default
import json

app = AgenticGraph().build_workflow().compile()
state3 = get_agent_state_default(
    chunk_index="miles_guo",
    max_iter=2, chunk_k=12, max_query_expand_k=2)
state3["question"] = "疫苗副作用"

# 只跑到 rag_search 就停，拿「未过滤」的 search_results，不继续 rag_reply / reply_validation
async def run_until_search_and_print():
    async for chunk in app.astream(state3.copy(), stream_mode="updates"):
        if "rag_search" in chunk:
            update = chunk["rag_search"]
            all_results = update.get("search_results", [])
            hist_ops = update.get("historical_search_ops", [])
            # 本轮请求的 top_k / query_list（便于核对是否用上 chunk_k=12）
            if hist_ops:
                last_op = hist_ops[-1]
                args = last_op.get("function", {}).get("arguments", "{}")
                try:
                    args_d = json.loads(args) if isinstance(args, str) else args
                    print("本轮搜索请求:", args_d)
                except Exception:
                    print("本轮搜索请求 (raw):", args)
            print("=== 原始搜索结果（未过滤，未走完所有 node）===")
            print(f"共 {len(all_results)} 条（期望接近 chunk_k=12，去重后可能略少）\n")
            for i, r in enumerate(all_results):
                print(f"[{i+1}]", json.dumps(r, ensure_ascii=False, indent=2))
            return  # 不继续执行 rag_reply 等后续节点
    print("未进入 rag_search（可能被分类为 greeting/unclear 等）")

await run_until_search_and_print()